[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_28/2_sql.ipynb)

# Day 28: SQL (mock exam)

**Mock exam rules (30 minutes, one timer for the whole notebook).**

- This is a company-style round: TikTok, Google and Samsung flavoured questions, mixed. Start one 30 minute timer.
- About 5 minutes a query. Before each query say your metric definition in one sentence (examiners grade that).
- No hints or solutions until the timer rings. Then grade: 1 point correct, 0.5 for right logic with a small slip.
  5 or more is exam ready. Put every miss in the mistake log; day 30 and the redo days bring them back.
- After grading, answer each follow-up out loud in under a minute.

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (31 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Fastest start per category (review) | hard | 5 |
| Q2 | How much do people click on the top results? | hard | 4 |
| Q3 | Firmware adoption curve by series (review) | hard | 5 |
| Q4 | The most divisive movies | hard | 5 |
| Q5 | Quick second orders at a real shop (review) | hard | 6 |
| Q6 | Longest 10k-step streak (after cleaning) (review) | hard | 6 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Tables

**Made-up short-video tables** (invented for practice, fixed seed; May 2024):

| Table | One row is | Columns |
|---|---|---|
| `tt_creators` | one creator (40) | `creator_id`, `category` (main category), `country` |
| `tt_videos` | one posted video | `video_id`, `creator_id`, `posted_at` ('YYYY-MM-DD HH:MM:SS'), `duration_s`, `category` |
| `tt_views` | one play of a video | `view_id`, `viewer_id`, `video_id`, `view_time`, `watch_s` (seconds watched; can be MORE than `duration_s` when the video loops), `source` (fyp / following) |
| `tt_follows` | one follow | `follower_id`, `creator_id`, `followed_at` |

**Made-up search tables** (invented for practice, fixed seed; one week from 2024-06-03):

| Table | One row is | Columns |
|---|---|---|
| `searches` | one query typed by a user | `search_id`, `user_id`, `search_time` ('YYYY-MM-DD HH:MM:SS'), `query`, `device` (mobile / desktop / tablet) |
| `clicks` | one click on a result | `search_id`, `position` (1 = top result), `click_time`, `dwell_s` (seconds on the page before coming back; > 30 counts as a "good click") |

**Made-up phone tables** (invented for practice, fixed seed):

| Table | One row is | Columns |
|---|---|---|
| `devices` | one phone a user activated | `device_id`, `user_id`, `model` (e.g. 'Galaxy S23'), `series` (S / A / Z), `activated_at` ('YYYY-MM-DD'). A user's next device replaces the previous one |
| `firmware_updates` | one firmware install | `device_id`, `version` ('6.0' or '6.1'), `installed_at` ('YYYY-MM-DD HH:MM:SS') |
| `crash_logs` | one app crash, May and June 2024 | `device_id`, `crash_time`, `app` |
| `health_steps` | one sync from a watch, June 2024 | `user_id`, `step_date` ('YYYY-MM-DD'), `steps` (the day's total SO FAR at sync time), `synced_at`. Several syncs per day; some syncs are uploaded twice |

**Real tables: MovieLens small** (GroupLens; 610 users rated 9,742 movies, 1996 to 2018):

| Table | One row is | Columns |
|---|---|---|
| `ml_ratings` | one rating (100,836) | `user_id`, `movie_id`, `rating` (0.5 to 5.0 in steps of 0.5), `rated_at` ('YYYY-MM-DD HH:MM:SS', UTC) |
| `ml_movies` | one movie (9,742) | `movie_id`, `title`, `year` (from the title, can be NULL), `genres` (pipe separated, e.g. 'Action|Comedy') |
| `ml_movie_genres` | one (movie, genre) pair | `movie_id`, `genre` (the `genres` string split into rows) |

**Real table: Online Retail** (UCI, a UK online gift shop, 2010-12-01 to 2011-12-09, CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (text; a 'C' prefix = cancellation), `stock_code`, `description`, `quantity` (negative for cancellations), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price` (GBP), `customer_id` (NULL for about 25% of rows), `country` |

The setup cell prints every table with its columns. Call `tables()` again any time.

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up short-video tables (generated with a fixed seed; numpy's legacy RandomState is frozen across versions).
import numpy as np

def make_video_tables(con, seed=25):
    rs = np.random.RandomState(seed)
    cats = ["dance", "food", "comedy", "education", "gaming"]
    nc = 40
    creators = pd.DataFrame({
        "creator_id": np.arange(1, nc + 1),
        "category": [cats[i] for i in rs.randint(0, 5, nc)],
        "country": [["US", "KR", "BR", "ID", "DE"][i] for i in rs.choice(5, nc, p=[.3, .25, .2, .15, .1])],
    })
    pop = rs.lognormal(0, 1.0, nc)                              # creator popularity
    start = pd.Timestamp("2024-05-01")
    vids = []
    vid = 1000
    for c in range(nc):
        for _ in range(1 + rs.poisson(6)):
            vid += 1
            posted = start + pd.Timedelta(seconds=int(rs.randint(0, 27 * 86400)))
            cat = creators.category[c] if rs.rand() < 0.85 else cats[rs.randint(0, 5)]
            vids.append((vid, c + 1, posted, int(rs.choice([15, 30, 45, 60, 90, 180])), cat))
    videos = pd.DataFrame(vids, columns=["video_id", "creator_id", "posted_at", "duration_s", "category"])
    videos = videos.sort_values("posted_at", kind="mergesort").reset_index(drop=True)
    vq = rs.beta(2, 2, len(videos))                             # video quality: how much of it people watch
    w = pop[videos.creator_id.values - 1] * (0.5 + vq)
    posted_s = ((videos.posted_at - start).dt.total_seconds()).values
    views, follows, followed = [], [], set()
    for u in range(1, 1201):
        n_sess = rs.poisson(4)
        for _ in range(n_sess):
            t = int(rs.randint(1 * 86400, 31 * 86400))
            n_vid = 1 + rs.geometric(1 / 6.0)
            source = "following" if rs.rand() < 0.15 else "fyp"
            for _ in range(n_vid):
                ok = np.nonzero(posted_s < t)[0]
                if len(ok) == 0:
                    break
                p = w[ok] / w[ok].sum()
                i = ok[rs.choice(len(ok), p=p)]
                dur = videos.duration_s[i]
                frac = min(rs.beta(1 + 4 * vq[i], 2), 1.0)
                if rs.rand() < 0.08:
                    frac += 1 + rs.rand()                       # rewatch loop: watched more than the length
                watch = max(1, int(round(frac * dur)))
                views.append((u, int(videos.video_id[i]), start + pd.Timedelta(seconds=t), watch, source))
                cid = int(videos.creator_id[i])
                if frac > 0.8 and (u, cid) not in followed and rs.rand() < 0.06:
                    followed.add((u, cid))
                    follows.append((u, cid, start + pd.Timedelta(seconds=t + watch + 2)))
                t += watch + int(rs.randint(1, 15))
                source = "fyp"
    fmt = "%Y-%m-%d %H:%M:%S"
    views = pd.DataFrame(views, columns=["viewer_id", "video_id", "view_time", "watch_s", "source"])
    views = views.sort_values(["view_time", "viewer_id"], kind="mergesort").reset_index(drop=True)
    views.insert(0, "view_id", np.arange(1, len(views) + 1))
    views["view_time"] = views["view_time"].dt.strftime(fmt)
    follows = pd.DataFrame(follows, columns=["follower_id", "creator_id", "followed_at"])
    follows = follows.sort_values("followed_at", kind="mergesort").reset_index(drop=True)
    follows["followed_at"] = follows["followed_at"].dt.strftime(fmt)
    videos["posted_at"] = videos["posted_at"].dt.strftime(fmt)
    for name, df in [("tt_creators", creators), ("tt_videos", videos), ("tt_views", views), ("tt_follows", follows)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_video_tables(db)

# Made-up web search log (generated with a fixed seed).
import numpy as np

def make_search_tables(con, seed=26):
    rs = np.random.RandomState(seed)
    topics = ["weather", "flights to seoul", "python list sort", "pizza near me", "nba scores", "tax deadline",
              "how to tie a tie", "galaxy s24 review", "bitcoin price", "sql window functions", "cheap hotels paris",
              "translate hello", "movie times", "covid symptoms", "best laptop 2024"]
    quality = rs.beta(4, 2, len(topics))                      # how often result 1 answers the query
    quality[[2, 9, 14]] = [0.25, 0.30, 0.20]                   # three hard queries
    devices = ["mobile", "desktop", "tablet"]
    start = pd.Timestamp("2024-06-03")
    s_rows, c_rows = [], []
    sid = 0
    for u in range(1, 801):
        dev = devices[rs.choice(3, p=[.6, .32, .08])]
        for _ in range(rs.poisson(5)):
            t = int(rs.randint(0, 7 * 86400))
            k = rs.randint(len(topics))
            q = topics[k]
            for attempt in range(3):
                sid += 1
                s_rows.append((sid, u, start + pd.Timedelta(seconds=t), q, dev))
                p_ok = quality[k] if attempt == 0 else min(quality[k] + 0.3, 0.95)
                if rs.rand() < p_ok:                           # satisfied: one long click near the top
                    pos = 1 if rs.rand() < 0.7 else int(rs.randint(2, 4))
                    c_rows.append((sid, pos, start + pd.Timedelta(seconds=t + int(rs.randint(2, 12))), int(rs.randint(31, 400))))
                    break
                if rs.rand() < 0.5:                            # short click, then back to the results
                    pos = int(rs.randint(1, 6))
                    c_rows.append((sid, pos, start + pd.Timedelta(seconds=t + int(rs.randint(2, 12))), int(rs.randint(2, 30))))
                    t += int(rs.randint(20, 60))
                if rs.rand() < 0.55:                           # reformulate the query a few seconds later
                    t += int(rs.randint(5, 50))
                    q = q + (" " + ["2024", "best", "how", "free", "near me"][rs.randint(5)])
                else:
                    break
    fmt = "%Y-%m-%d %H:%M:%S"
    s = pd.DataFrame(s_rows, columns=["search_id", "user_id", "search_time", "query", "device"])
    s["search_time"] = s["search_time"].dt.strftime(fmt)
    c = pd.DataFrame(c_rows, columns=["search_id", "position", "click_time", "dwell_s"])
    c["click_time"] = c["click_time"].dt.strftime(fmt)
    s.to_sql("searches", con, index=False, if_exists="replace")
    c.to_sql("clicks", con, index=False, if_exists="replace")

make_search_tables(db)

# Made-up phone fleet tables (generated with a fixed seed).
import numpy as np

def make_device_tables(con, seed=27):
    rs = np.random.RandomState(seed)
    models = [("Galaxy S21", "S", 21), ("Galaxy S22", "S", 22), ("Galaxy S23", "S", 23), ("Galaxy S24", "S", 24),
              ("Galaxy A52", "A", 52), ("Galaxy A53", "A", 53), ("Galaxy A54", "A", 54),
              ("Galaxy Z Flip4", "Z", 4), ("Galaxy Z Flip5", "Z", 5)]
    release = {"Galaxy S21": "2021-01-29", "Galaxy S22": "2022-02-25", "Galaxy S23": "2023-02-17",
               "Galaxy S24": "2024-01-31", "Galaxy A52": "2021-03-26", "Galaxy A53": "2022-03-24",
               "Galaxy A54": "2023-03-24", "Galaxy Z Flip4": "2022-08-26", "Galaxy Z Flip5": "2023-08-11"}
    end = pd.Timestamp("2024-06-30")
    dev_rows, did = [], 0
    for u in range(1, 701):
        m = models[rs.choice(len(models))]
        t = pd.Timestamp(release[m[0]]) + pd.Timedelta(days=int(rs.randint(0, 200)))
        while t <= end:
            did += 1
            dev_rows.append((did, u, m[0], m[1], t))
            nxt = [x for x in models if x[1] == m[1] and x[2] > m[2]]
            if rs.rand() < 0.15:                                # switches line
                nxt = [x for x in models if x[1] != m[1] and pd.Timestamp(release[x[0]]) > t]
            if not nxt or rs.rand() < 0.35:
                break
            m = nxt[rs.randint(len(nxt))]
            t = max(t + pd.Timedelta(days=int(rs.randint(150, 900))), pd.Timestamp(release[m[0]]))
    devices = pd.DataFrame(dev_rows, columns=["device_id", "user_id", "model", "series", "activated_at"])
    # firmware: everybody active in May 2024 gets v6.0 then v6.1 (v6.1 has a crash bug on Galaxy A54 and Z Flip5)
    fw, cr = [], []
    active = devices[devices.activated_at < pd.Timestamp("2024-05-01")]
    nxt_act = devices.groupby("user_id")["activated_at"].shift(-1)
    for i, d in active.iterrows():
        if pd.notna(nxt_act[i]) and nxt_act[i] < pd.Timestamp("2024-05-01"):
            continue                                            # replaced before May: not in use any more
        up = pd.Timestamp("2024-05-01") + pd.Timedelta(seconds=int(rs.randint(0, 25 * 86400)))
        fw.append((d.device_id, "6.0", pd.Timestamp("2024-04-01") + pd.Timedelta(days=int(rs.randint(0, 20)))))
        took_update = rs.rand() < 0.8
        if took_update:
            fw.append((d.device_id, "6.1", up))
        base = {"S": 0.04, "A": 0.07, "Z": 0.09}[d.series]
        for day in range(61):
            ts = pd.Timestamp("2024-05-01") + pd.Timedelta(days=day, seconds=int(rs.randint(0, 86400)))
            rate = base
            if took_update and ts >= up and d.model in ("Galaxy A54", "Galaxy Z Flip5"):
                rate = base * 3.5
            if rs.rand() < rate:
                cr.append((d.device_id, ts, ["camera", "launcher", "messages", "gallery"][rs.randint(4)]))
    fmt = "%Y-%m-%d %H:%M:%S"
    devices["activated_at"] = devices["activated_at"].dt.strftime("%Y-%m-%d")
    fw = pd.DataFrame(fw, columns=["device_id", "version", "installed_at"])
    fw["installed_at"] = pd.to_datetime(fw["installed_at"]).dt.strftime(fmt)
    fw = fw.sort_values(["installed_at", "device_id"], kind="mergesort").reset_index(drop=True)
    cr = pd.DataFrame(cr, columns=["device_id", "crash_time", "app"])
    cr["crash_time"] = pd.to_datetime(cr["crash_time"]).dt.strftime(fmt)
    cr = cr.sort_values(["crash_time", "device_id"], kind="mergesort").reset_index(drop=True)
    # health steps for users 1..60 in June 2024: the watch syncs several times a day; each sync sends the day's total so far
    hs = []
    for u in range(1, 61):
        level = rs.randint(4000, 13000)
        for day in range(30):
            if rs.rand() < 0.08:
                continue                                        # watch not worn
            total = max(300, int(rs.normal(level, 3500)))
            n_sync = 1 + rs.poisson(1.5)
            cuts = np.sort(rs.rand(n_sync))
            cuts[-1] = 1.0
            for k, f in enumerate(cuts):
                ts = pd.Timestamp("2024-06-01") + pd.Timedelta(days=day, hours=8 + int(14 * (k + 1) / n_sync), minutes=int(rs.randint(0, 59)))
                hs.append((u, (pd.Timestamp("2024-06-01") + pd.Timedelta(days=day)).strftime("%Y-%m-%d"), int(total * f), ts.strftime(fmt)))
                if rs.rand() < 0.05:
                    hs.append(hs[-1])                           # duplicate upload of the same sync
    hs = pd.DataFrame(hs, columns=["user_id", "step_date", "steps", "synced_at"])
    for name, df in [("devices", devices), ("firmware_updates", fw), ("crash_logs", cr), ("health_steps", hs)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_device_tables(db)

# Real data: MovieLens small (GroupLens, research and education use). Downloads the zip if the CSVs are missing.
import zipfile
if not os.path.exists(os.path.join(DATA, "movielens_ratings.csv")):
    _z = data_path("ml-latest-small.zip", "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip")
    with zipfile.ZipFile(_z) as zf:
        for _n in ("ratings", "movies"):
            with open(os.path.join(DATA, f"movielens_{_n}.csv"), "wb") as fh:
                fh.write(zf.read(f"ml-latest-small/{_n}.csv"))
_r = pd.read_csv(data_path("movielens_ratings.csv"))
_r["rated_at"] = pd.to_datetime(_r["timestamp"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S")
_r = _r.rename(columns={"userId": "user_id", "movieId": "movie_id"})[["user_id", "movie_id", "rating", "rated_at"]]
_r.to_sql("ml_ratings", db, index=False)
_m = pd.read_csv(data_path("movielens_movies.csv")).rename(columns={"movieId": "movie_id"})
_m["year"] = pd.to_numeric(_m["title"].str.extract(r"\((\d{4})\)\s*$")[0], errors="coerce").astype("Int64")
_m[["movie_id", "title", "year", "genres"]].to_sql("ml_movies", db, index=False)
_g = _m.assign(genre=_m["genres"].str.split("|")).explode("genre")[["movie_id", "genre"]]
_g[_g["genre"] != "(no genres listed)"].to_sql("ml_movie_genres", db, index=False)

# Real data: UCI Online Retail (a UK gift shop, 2010-12 to 2011-12, CC BY 4.0).
import io, zipfile
_p = os.path.join(DATA, "online_retail.csv.gz")
if not os.path.exists(_p):                       # first run in Colab: download the UCI zip and convert it once
    with zipfile.ZipFile(data_path("online_retail.zip", "https://archive.ics.uci.edu/static/public/352/online+retail.zip")) as zf:
        pd.read_excel(io.BytesIO(zf.read("Online Retail.xlsx"))).to_csv(_p, index=False, compression="gzip")
_o = pd.read_csv(_p, dtype={"InvoiceNo": str, "StockCode": str})
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id", "country"]
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o.to_sql("retail", db, index=False)

def tables():
    """Prints every table with its row count and columns."""
    for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
        cols = [r[1] for r in db.execute(f'PRAGMA table_info("{t}")')]
        n = db.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
        print(f"{t} ({n:,} rows): {', '.join(cols)}")

tables()

---
### Q1. Fastest start per category (review)

*hard, about 5 min*

TikTok style. For each video count the views in its **first 24 hours** after `posted_at` (videos with no
views count 0). Return the video with the most first-day views in each category (`tt_videos.category`): `category`,
`video_id`, `creator_id`, `views_24h`, ordered by `views_24h` descending. Ties: show all tied videos.

*Follow-up:* why measure the first 24 hours instead of total views?

*Source: [DataLemur TikTok SQL questions (inspiration)](https://datalemur.com/blog/tiktok-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: a time window relative to each row's own start, then top 1 per group. Pattern: LEFT JOIN with the window in the ON clause, then RANK per category.

</details>

<details><summary><b>Hint 2</b></summary>

1. `tt_videos v LEFT JOIN tt_views w ON w.video_id = v.video_id AND w.view_time < datetime(v.posted_at, '+24 hours')`.
2. COUNT(w.view_id) per video.
3. `RANK() OVER (PARTITION BY category ORDER BY views_24h DESC)`, keep 1.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH first_day AS (
  SELECT v.video_id, v.category, v.creator_id, COUNT(w.view_id) AS views_24h
  FROM tt_videos v
  LEFT JOIN tt_views w
    ON w.video_id = v.video_id
   AND w.view_time < datetime(v.posted_at, '+24 hours')
  GROUP BY v.video_id, v.category, v.creator_id
), r AS (
  SELECT *, RANK() OVER (PARTITION BY category ORDER BY views_24h DESC) AS rk
  FROM first_day
)
SELECT category, video_id, creator_id, views_24h
FROM r
WHERE rk = 1
ORDER BY views_24h DESC
```

Result:

| category | video_id | creator_id | views_24h |
|---|---|---|---|
| food | 1108 | 16 | 397 |
| gaming | 1279 | 38 | 107 |
| dance | 1028 | 5 | 71 |
| comedy | 1285 | 39 | 66 |
| education | 1117 | 17 | 46 |

**Why:** The window condition belongs in the ON clause: in WHERE it would remove videos without early views. RANK keeps ties as asked. Food video 1108 by creator 16 gets 397 views on day one. Follow-up: total views favour old videos (more time to collect views); a fixed window compares videos posted on different days fairly, and early velocity is what a recommender uses to decide whether to push a video wider.

**Complexity:** One join (views filtered per video) and one window over about 300 videos.

**Common mistakes:** COUNT(*) after a LEFT JOIN (a video with no views counts 1). Window measured from the first VIEW, not the post time.

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q2. How much do people click on the top results?

*hard, about 4 min*

Google style. For each result `position` in `clicks` return `clicks`, `pct` (share of all clicks) and
`cum_pct` (share of clicks at this position or higher up), 1 decimal, ordered by position.

Example: positions 1, 2, 3 with 60, 30, 10 clicks: cum_pct = 60.0, 90.0, 100.0.

*Source: [DataLemur Google SQL questions (inspiration)](https://datalemur.com/blog/google-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'share' and 'this position or above'. Pattern: running total with SUM OVER (ORDER BY position) divided by the grand total SUM OVER ().

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: COUNT(*) per position.
2. `n / SUM(n) OVER ()` and `SUM(n) OVER (ORDER BY position ROWS UNBOUNDED PRECEDING) / SUM(n) OVER ()`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH p AS (
  SELECT position, COUNT(*) AS clicks FROM clicks GROUP BY position
)
SELECT position, clicks,
  ROUND(100.0 * clicks / SUM(clicks) OVER (), 1) AS pct,
  ROUND(100.0 * SUM(clicks) OVER (ORDER BY position ROWS UNBOUNDED PRECEDING)
        / SUM(clicks) OVER (), 1) AS cum_pct
FROM p
ORDER BY position
```

Result:

| position | clicks | pct | cum_pct |
|---|---|---|---|
| 1 | 2408 | 60.0 | 60.0 |
| 2 | 645 | 16.1 | 76.0 |
| 3 | 622 | 15.5 | 91.5 |
| 4 | 170 | 4.2 | 95.8 |
| 5 | 170 | 4.2 | 100.0 |

**Why:** Window functions run on the grouped rows, so one SELECT gives both the share and the cumulative share. 60.0% of clicks go to position 1 and 91.5% to the top 3, which is why ranking metrics weight the top heavily.

**Complexity:** One aggregation, one window over 5 rows.

**Common mistakes:** Integer division (`clicks / SUM(...)` = 0). Cumulating in the wrong direction (ORDER BY position DESC).

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q3. Firmware adoption curve by series (review)

*hard, about 5 min*

Samsung style. The devices that received firmware 6.0 are the active fleet. Per `series` (S / A / Z) return
`devices`, and the % of them that had installed 6.1 **before** 2024-05-08 (`by_may7`), before 2024-05-15 (`by_may14`),
at any time (`ever`), and never (`never`), 1 decimal. Order by series.

*Follow-up:* product wants to compare crash rates of updated and not-updated phones to judge 6.1. What is the problem?

*Source: [DataLemur Samsung SQL questions (inspiration)](https://datalemur.com/blog/samsung-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: shares at several cut-off dates as columns. Pattern: LEFT JOIN to the first install time, then a pivot with conditional aggregation.

</details>

<details><summary><b>Hint 2</b></summary>

1. `base`: DISTINCT device_id, series for version 6.0.
2. `up`: MIN(installed_at) of 6.1 per device.
3. `base LEFT JOIN up`; `100.0 * SUM(CASE WHEN up_t < '2024-05-08' THEN 1 ELSE 0 END) / COUNT(*)` etc.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH base AS (
  SELECT DISTINCT f.device_id, d.series
  FROM firmware_updates f JOIN devices d ON d.device_id = f.device_id
  WHERE f.version = '6.0'
), up AS (
  SELECT device_id, MIN(installed_at) AS up_t
  FROM firmware_updates WHERE version = '6.1'
  GROUP BY device_id
)
SELECT b.series, COUNT(*) AS devices,
  ROUND(100.0 * SUM(CASE WHEN u.up_t < '2024-05-08' THEN 1 ELSE 0 END) / COUNT(*), 1) AS by_may7,
  ROUND(100.0 * SUM(CASE WHEN u.up_t < '2024-05-15' THEN 1 ELSE 0 END) / COUNT(*), 1) AS by_may14,
  ROUND(100.0 * SUM(CASE WHEN u.up_t IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS ever,
  ROUND(100.0 * SUM(CASE WHEN u.up_t IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS never
FROM base b
LEFT JOIN up u ON u.device_id = b.device_id
GROUP BY b.series
ORDER BY b.series
```

Result:

| series | devices | by_may7 | by_may14 | ever | never |
|---|---|---|---|---|---|
| A | 216 | 21.3 | 43.5 | 79.2 | 20.8 |
| S | 276 | 23.6 | 43.8 | 77.5 | 22.5 |
| Z | 156 | 25.0 | 44.2 | 76.9 | 23.1 |

**Why:** The LEFT JOIN keeps devices that never updated (up_t NULL), which is the 'never' column (about 21% to 23%). Text timestamps compare correctly with `<` because the format is fixed width. Adoption is almost the same across series (about 44% by May 14). Follow-up: updaters self-select (heavier users, newer phones, Wi-Fi users), so their crash rate differs for reasons other than the firmware. Compare each device with itself before and after (as on day 27) and use the never-updated devices as a calendar control.

**Complexity:** Two small aggregations and one join.

**Common mistakes:** Inner join (never-updated devices vanish, 'ever' becomes 100%). Using `<=` with a date string against a timestamp (`'2024-05-07 10:00' <= '2024-05-07'` is false).

**Learn more:** [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls)

</details>

---
### Q4. The most divisive movies

*hard, about 5 min*

MovieLens (real). Among movies with at least 50 ratings, find the 5 with the **highest variance** of
ratings (the most divisive). Return `title`, `n`, `avg_rating` (2 decimals) and `var` (population variance, 2
decimals), highest variance first. Your SQL engine has no STDDEV or SQRT function.

Example: ratings 1 and 5: mean 3, variance = ((1-3)^2 + (5-3)^2) / 2 = 4.

*Follow-up:* why is a high-variance movie interesting for a recommender?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: spread without STDDEV. Pattern: aggregation with the identity `var = AVG(x * x) - AVG(x) * AVG(x)`.

</details>

<details><summary><b>Hint 2</b></summary>

1. Join ratings to movies, GROUP BY movie, HAVING COUNT(*) >= 50.
2. var = `AVG(rating * rating) - AVG(rating) * AVG(rating)`.
3. ORDER BY var DESC LIMIT 5.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT m.title, COUNT(*) AS n,
  ROUND(AVG(r.rating), 2) AS avg_rating,
  ROUND(AVG(r.rating * r.rating) - AVG(r.rating) * AVG(r.rating), 2) AS var
FROM ml_ratings r
JOIN ml_movies m ON m.movie_id = r.movie_id
GROUP BY r.movie_id, m.title
HAVING COUNT(*) >= 50
ORDER BY var DESC
LIMIT 5
```

Result:

| title | n | avg_rating | var |
|---|---|---|---|
| Blair Witch Project, The (1999) | 64 | 2.8 | 1.86 |
| First Knight (1995) | 54 | 3.08 | 1.66 |
| Austin Powers: The Spy Who Shagged Me (1999) | 121 | 3.2 | 1.46 |
| Scream (1996) | 70 | 3.2 | 1.41 |
| Moulin Rouge (2001) | 55 | 3.59 | 1.41 |

**Why:** `E[x^2] - E[x]^2` is the population variance, computable in one pass with plain aggregates (the sample variance multiplies by n / (n - 1)). The Blair Witch Project tops the list (variance 1.86, so the standard deviation is about 1.36). The 50 rating minimum stops movies with 2 ratings (1 and 5) from winning. Follow-up: the average hides that some users love and others hate the movie; a personalised model can separate the two groups, while a popularity ranking by average would treat it as mediocre.

**Complexity:** One join and one GROUP BY.

**Common mistakes:** Ranking by the range MAX - MIN (almost always 0.5 to 5). Forgetting the minimum count. Integer math (not an issue here, ratings are real numbers). PostgreSQL has `VAR_POP` and `STDDEV_SAMP`.

**Learn more:** [sql-aggregation](https://mahsa-agz.github.io/ds-handbook/#sql-aggregation), [sql-median-percentiles](https://mahsa-agz.github.io/ds-handbook/#sql-median-percentiles)

</details>

---
### Q5. Quick second orders at a real shop (review)

*hard, about 6 min*

UCI Online Retail (real). An order is one `invoice_no` (real sales only: customer known, not 'C', quantity
> 0); its time is the earliest `invoice_date` of its lines. For customers whose **first** order was before
2011-11-01, compute the % who placed a second order within 30 days of the first. Return per first-order month:
`first_month`, `customers`, `pct_30d` (1 decimal).

*Follow-up:* why the cut-off at 2011-11-01?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: 'first order' and 'the next order within N days'. Pattern: ROW_NUMBER to find the first order and LEAD for the next one (retention of buyers).

</details>

<details><summary><b>Hint 2</b></summary>

1. `orders`: GROUP BY customer_id, invoice_no with MIN(invoice_date).
2. `ROW_NUMBER()` and `LEAD(t)` per customer ordered by t, invoice_no.
3. Keep rn = 1 and t < '2011-11-01'; flag next_t - t <= 30 days; AVG per month.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH orders AS (
  SELECT customer_id, invoice_no, MIN(invoice_date) AS t
  FROM retail
  WHERE customer_id IS NOT NULL AND invoice_no NOT LIKE 'C%' AND quantity > 0
  GROUP BY customer_id, invoice_no
), seq AS (
  SELECT customer_id, t,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY t, invoice_no) AS rn,
    LEAD(t) OVER (PARTITION BY customer_id ORDER BY t, invoice_no) AS next_t
  FROM orders
)
SELECT substr(t, 1, 7) AS first_month, COUNT(*) AS customers,
  ROUND(100.0 * AVG(CASE WHEN next_t IS NOT NULL
                          AND julianday(next_t) - julianday(t) <= 30 THEN 1 ELSE 0 END), 1) AS pct_30d
FROM seq
WHERE rn = 1 AND t < '2011-11-01'
GROUP BY substr(t, 1, 7)
ORDER BY first_month
```

Result:

| first_month | customers | pct_30d |
|---|---|---|
| 2010-12 | 885 | 35.0 |
| 2011-01 | 417 | 20.4 |
| 2011-02 | 380 | 16.1 |
| 2011-03 | 452 | 16.6 |
| 2011-04 | 300 | 16.0 |
| 2011-05 | 284 | 18.0 |
| 2011-06 | 242 | 17.4 |
| 2011-07 | 188 | 21.8 |
| 2011-08 | 169 | 14.8 |
| 2011-09 | 299 | 18.4 |
| 2011-10 | 358 | 24.0 |

**Why:** Orders are built first (one row per invoice), so a 30 line invoice is one order. LEAD on the first row gives the second order. December 2010 looks best (35.0%) because it mixes in old loyal customers who bought before the data starts; later months sit around 15% to 24%. Follow-up: the data ends on 2011-12-09; a customer whose first order was on 2011-11-20 had only 19 days to come back, so recent months would look worse for no real reason (right censoring).

**Complexity:** Aggregation to orders, then one window sort per customer.

**Common mistakes:** Working on invoice LINES (the 'next line' is usually the same invoice: gap 0). Ignoring the censoring cut-off. Counting cancellations as orders.

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead)

</details>

---
### Q6. Longest 10k-step streak (after cleaning) (review)

*hard, about 6 min*

Samsung Health style. `health_steps` has several syncs per day (cumulative counts) and duplicate uploads;
the day's value is the latest sync. A **10k day** is a day with at least 10,000 steps. For each user find their
longest run of consecutive 10k days and return the top 5 runs overall: `user_id`, `start_date`, `end_date`, `days`
(longest first, then user_id). A user can appear twice if they have two long runs.

*Follow-up:* what goes wrong if you skip the cleaning step?

*Source: [DataLemur Samsung SQL questions (inspiration)](https://datalemur.com/blog/samsung-sql-interview-questions)*

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: dirty snapshots, then 'consecutive days'. Pattern: dedup with ROW_NUMBER (latest sync), then gaps and islands (date minus row number).

</details>

<details><summary><b>Hint 2</b></summary>

1. `final`: rn = 1 of `ROW_NUMBER() OVER (PARTITION BY user_id, step_date ORDER BY synced_at DESC, steps DESC)`.
2. `good`: 10k days with `julianday(step_date) - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY step_date)`.
3. GROUP BY user, group key: MIN, MAX, COUNT; ORDER BY days DESC, user_id LIMIT 5.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH ranked AS (
  SELECT user_id, step_date, steps,
    ROW_NUMBER() OVER (PARTITION BY user_id, step_date ORDER BY synced_at DESC, steps DESC) AS rn
  FROM health_steps
), good AS (
  SELECT user_id, step_date,
    julianday(step_date) - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY step_date) AS grp
  FROM ranked
  WHERE rn = 1 AND steps >= 10000
)
SELECT user_id, MIN(step_date) AS start_date, MAX(step_date) AS end_date, COUNT(*) AS days
FROM good
GROUP BY user_id, grp
ORDER BY days DESC, user_id
LIMIT 5
```

Result:

| user_id | start_date | end_date | days |
|---|---|---|---|
| 31 | 2024-06-04 | 2024-06-14 | 11 |
| 14 | 2024-06-12 | 2024-06-21 | 10 |
| 23 | 2024-06-09 | 2024-06-17 | 9 |
| 31 | 2024-06-16 | 2024-06-24 | 9 |
| 44 | 2024-06-01 | 2024-06-08 | 8 |

**Why:** The 10k filter is applied to the cleaned daily value, BEFORE numbering the rows, so only 10k days take part in the date minus row number trick. User 31 has the longest run (11 days, June 4 to 14) and also a 9 day run later. Follow-up: on the raw table a day has several rows, so the row number grows faster than the date and streaks break apart, and early syncs below 10k make the day look like a gap; with the raw rows, user 60 even looks like the best walker because its duplicate rows stretch one streak to 20 rows.

**Complexity:** Two window sorts: O(n log n).

**Common mistakes:** Filtering 10k on the raw syncs (an early sync of 4,000 on a 12,000 day). Running ROW_NUMBER before the 10k filter (then non-10k days fill the gaps).

**Learn more:** [sql-gaps-islands](https://mahsa-agz.github.io/ds-handbook/#sql-gaps-islands), [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_28/3_stats.ipynb)